# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all available record sets and fields by their `@id` as defined in the Croissant schema.

In [ ]:
# List all record sets and fields with their @id
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']} - {rs.get('name', '')}")
        fields = rs.get('field', [])
        for f in fields:
            print(f"  Field: {f['@id']} - {f.get('name', '')}")
        print("")
# If actual record sets are not present in metadata, try listing from the dataset directly
if not record_sets:
    try:
        all_record_set_ids = dataset.record_set_ids
        if all_record_set_ids:
            print("RecordSet IDs from dataset:")
            for rset_id in all_record_set_ids:
                print(f"- {rset_id}")
    except AttributeError:
        print("Unable to automatically list record sets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

To extract data, we need the record set `@id`. Below, we use the typical idiom for loading all record sets found.

In [ ]:
# Attempt to detect available record sets dynamically
try:
    record_set_ids = dataset.record_set_ids
    print("Available record set @ids:", record_set_ids)
except AttributeError:
    record_set_ids = []
    print("Record sets could not be listed automatically.")

# If none found previously, manually define a placeholder
if not record_set_ids:
    # You may need to replace this with the actual @id if discovered from metadata
    record_set_ids = ['http://mlcommons.org/croissant/RecordSet/main_surveys']

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records from RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Fields (columns) in {record_set_id}:", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Could not retrieve records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we select a numeric mental health score field (e.g., `PHQ-9` score) and perform some basic EDA.

In [ ]:
# Select the first available record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to pick likely numeric fields such as 'phq9_score' or 'gad7_score' or 'psq_score'
    possible_numeric_fields = [
        field for field in df.columns
        if ('phq9' in field.lower() or 'gad7' in field.lower() or 'psq' in field.lower()) and df[field].dtype in [int, float, 'int64', 'float64']
    ]
    
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id}")
    else:
        numeric_field_id = df.select_dtypes(include=['number']).columns.tolist()
        numeric_field_id = numeric_field_id[0] if numeric_field_id else df.columns[0]
        print(f"Defaulting to first numeric field @id: {numeric_field_id}")

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to use a group field: for example 'village', 'gender', or 'age_group'
    group_field = None
    candidate_group_fields = [
        col for col in df.columns if ('village' in col.lower() or 'gender' in col.lower() or 'education' in col.lower() or 'marital' in col.lower())
    ]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Grouping by {group_field}")

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, histogram or boxplot of a mental health score, or bar chart by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Plot distribution of chosen numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field was found, show mean scores by group
    if group_field:
        means = df.groupby(group_field)[numeric_field_id].mean()
        plt.figure(figsize=(10, 5))
        means.sort_values().plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich demographic and mental health scores for Kilifi County, Kenya.
- Using the Croissant schema, we can identify `@id` relationships for reproducible querying and processing.
- Sample EDA showed distributions and grouped statistics, supporting data-driven conclusions about mental health indicators by demographic attributes.

For advanced use, see [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant) and the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).